In [22]:
from transformers import ViTForImageClassification, ViTImageProcessor
from PIL import Image
import requests
import torch
import time

url = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTPj25xraTPGV0ioxvc8qVx-E8QvN5uQqODHqK83Qr5fh3aaAyZ5OoMiOg&s=10"

image = Image.open(
    requests.get(url, stream=True).raw
).convert("RGB")

model = ViTForImageClassification.from_pretrained(
    "google/vit-large-patch16-224"
)

feature_extractor = ViTImageProcessor.from_pretrained(
    "google/vit-large-patch16-224"
)

inputs = feature_extractor(
    images=image,
    return_tensors="pt"
)

ti = time.time()

model.eval()

with torch.no_grad():
    outputs = model(**inputs)
    probs = torch.nn.functional.softmax(
        outputs.logits,
        dim=1
    )

tf = time.time()
tt = tf - ti

top_probs, top_preds = torch.topk(probs, k=5)

for prob, pred in zip(top_probs[0], top_preds[0]):
    label = model.config.id2label[pred.item()]
    print(
        f"Classe: {label} | Probabilidade: {prob.item():.4f}"
    )

print(f"\nTempo de inferência: {tt:.4f} segundos")





Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

Classe: wombat | Probabilidade: 0.9993
Classe: koala, koala bear, kangaroo bear, native bear, Phascolarctos cinereus | Probabilidade: 0.0000
Classe: badger | Probabilidade: 0.0000
Classe: mink | Probabilidade: 0.0000
Classe: echidna, spiny anteater, anteater | Probabilidade: 0.0000

Tempo de inferência: 3.6749 segundos
